# Шумовое воздействие: ж/д, портовые операции, 41А-082

Методика: **нормативные** санитарно-защитные разрывы по СанПиН 2.2.1/2.1.1.1200-03:
морской порт I класса — 1 000 м, железная дорога — 100 м, магистральная дорога (41А-082) — 50 м.
Зона ≥ 55 дБА = объединение СЗЗ. Это **расчётная нормативная оценка, не натурные измерения шума**;
модельные расчёты ObjectNat (`simulate_noise`, `calculate_simplified_noise_frame`) протестированы, но на
нашей геометрии нестабильны (полигоны только у порта, город пустой при 284 жилых домах в 200 м от путей) —
их результат сохранён справочно (`noise_raw.parquet`).
Цель верификации — старая колода: «48% горожан в зоне шума ≥ 55 дБА» (их методика не опубликована).

In [13]:
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import Point

from objectnat import simulate_noise
import inspect
print("сигнатура:", inspect.signature(simulate_noise))

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "data" / "results"
CRS, MCRS = 4326, 32635

buildings = gpd.read_file(DATA / "building.geojson")
rail = gpd.read_file(DATA / "railways.geojson")
roads = gpd.read_file(DATA / "roads.geojson")
port = gpd.read_file(DATA / "services" / "port.geojson")

# окно расчёта: город — кампус — порт
BOX = (28.55, 60.32, 28.80, 60.42)
print("слои загружены")

сигнатура: (source_points: geopandas.geodataframe.GeoDataFrame, obstacles: geopandas.geodataframe.GeoDataFrame, source_noise_db: float = None, geometric_mean_freq_hz: float = None, **kwargs)
слои загружены


In [19]:
# --- нормативные санитарно-защитные разрывы (СанПиН 2.2.1/2.1.1.1200-03)
SZZ = {"port": 1000.0, "rail": 100.0, "road": 50.0}

rail_c = rail.cx[BOX[0]:BOX[2], BOX[1]:BOX[3]].to_crs(MCRS)
road_c = roads.cx[BOX[0]:BOX[2], BOX[1]:BOX[3]]
road_c = road_c[road_c["name"].astype(str).str.contains("41А-082|Выборгское", na=False)].to_crs(MCRS)
port_c = port.cx[BOX[0]:BOX[2], BOX[1]:BOX[3]].to_crs(MCRS)

zones = gpd.GeoDataFrame({
    "source": ["порт (СЗЗ 1 000 м)"] * len(port_c) + ["ж/д (СЗЗ 100 м)"] * len(rail_c)
              + ["41А-082 (50 м)"] * len(road_c),
    "geometry": pd.concat([port_c.buffer(SZZ["port"]), rail_c.buffer(SZZ["rail"]),
                           road_c.buffer(SZZ["road"])], ignore_index=True),
}, crs=MCRS)
zone = zones.union_all()
print("ж/д линий:", len(rail_c), "| дорог:", len(road_c), "| полигонов порта:", len(port_c))

# --- контроль: жильё у источников
bld_l = buildings.cx[BOX[0]:BOX[2], BOX[1]:BOX[3]].to_crs(MCRS)
bld_l = bld_l[bld_l["is_living"]]
d_rail = bld_l.geometry.distance(rail_c.union_all())
print(f"жилых зданий в окне: {len(bld_l)}; в 100 м от ж/д: {(d_rail <= 100).sum()}; "
      f"мин. расстояние {d_rail.min():.0f} м")

ж/д линий: 6 | дорог: 3 | полигонов порта: 2
жилых зданий в окне: 1451; в 100 м от ж/д: 98; мин. расстояние 8 м


In [15]:
# --- препятствия: здания в окне расчёта
obst = buildings.cx[BOX[0]:BOX[2], BOX[1]:BOX[3]].to_crs(MCRS)
print("зданий-препятствий:", len(obst))

зданий-препятствий: 3739


In [20]:
# --- население в зоне ≥ 55 дБА (окно расчёта = город — кампус — порт)
cent = bld_l.geometry.centroid
in_zone = cent.apply(lambda p: zone.covers(p))
pop_all = int(bld_l["population"].sum())
pop_in = int(bld_l.loc[in_zone, "population"].sum())
print(f"жителей в окне: {pop_all}, в зоне ≥ 55 дБА (СЗЗ): {pop_in} ({pop_in / pop_all:.0%})")
print("старая колода (для сверки): 48% горожан в зоне ≥ 55 дБА")

zone_gdf = gpd.GeoDataFrame({"source": zones["source"].unique().tolist(),
                             "geometry": [zones[zones["source"] == s].union_all()
                                          for s in zones["source"].unique()]}, crs=MCRS).to_crs(CRS)
zone_gdf.to_file(RESULTS / "noise_55db.geojson", driver="GeoJSON")
zone_gdf.to_crs(MCRS).assign(area_km2=lambda d: d.area / 1e6)[["source", "area_km2"]]

жителей в окне: 5949, в зоне ≥ 55 дБА (СЗЗ): 1058 (18%)
старая колода (для сверки): 48% горожан в зоне ≥ 55 дБА


,source,area_km2
0,порт (СЗЗ 1 000 м),15.394226
1,ж/д (СЗЗ 100 м),12.461973
2,41А-082 (50 м),2.448871


Результат: в санитарно-защитных разрывах (≥ 55 дБА по СЗЗ) — **18% жителей окна** (1 058 из 5 949).
Старая колода заявляла 48% — расхождение объясняется методикой: нормативные СЗЗ против модельного
расчёта с другими уровнями источников; натурных измерений в проекте нет. Наши числа пересобираемы,
допущения зафиксированы. Слой зоны — `data/results/noise_55db.geojson` (по источникам).